# Part 9: Inference & Decoding — How Models Actually Generate

Training teaches a model to predict the next token. **Inference** is where we put
that prediction to work: turning a probability distribution over the vocabulary
into actual text, one token at a time.

This notebook is about the *decoding* half of a language model — the part that
runs every time you hit "send" on a chatbot. We will:

1. Recap **autoregressive generation** — the feed-back loop at the heart of every GPT.
2. Implement the major **decoding strategies from scratch** (greedy, temperature,
   top-k, top-p/nucleus, repetition penalty, beam search) and compare them on the
   *same* model and prompt.
3. Diagnose the **efficiency problem**: naive generation is O(n²) in the sequence length.
4. Build a **KV cache** from scratch and benchmark the speedup — this is the centerpiece.
5. Survey production tricks: batching, quantization, speculative decoding, GQA/MQA.
6. Measure **perplexity** and discuss why it is *not* the same as generation quality.

Everything operates on raw `logits = model(idx)[0][:, -1, :]` so you can see the
mechanics, rather than hiding behind `model.generate`.

---

In [ ]:
import sys
sys.path.append('..')

import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.gpt import GPT, create_gpt_small
from src.train import CharTokenizer, TextDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
%matplotlib inline

print(f"Using device: {device}")
print(f"torch {torch.__version__}")

## Step 0: Train a tiny model to generate from

To study *decoding*, we need a model whose next-token distribution is at least
mildly meaningful (not uniform noise). We train a small `create_gpt_small` for a
handful of epochs on the Shakespeare sample.

This is deliberately tiny and short — the goal is **decoding behavior**, not
literary quality. Expect the output to be Shakespeare-flavored gibberish. We train
this model **once** and reuse it for every demo below.

In [ ]:
# --- Data ---
with open('../data/sample_text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

tokenizer = CharTokenizer(text)
vocab_size = tokenizer.vocab_size
print(f"Characters: {len(text):,}  |  Vocab size: {vocab_size}")

# Hold out the last 10% of the corpus so perplexity is measured on text the
# model did NOT train on. Train on the prefix.
split = int(0.9 * len(text))
train_text, heldout_text = text[:split], text[split:]

# Small seq_len keeps training fast; this is also the model's max context.
SEQ_LEN = 64
dataset = TextDataset(train_text, tokenizer, SEQ_LEN)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
print(f"Training samples: {len(dataset):,}  |  Batches/epoch: {len(loader)}")

In [ ]:
# --- Train a small GPT for a few epochs (just enough to be non-random) ---
# NOTE: this dataset is tiny, so the model overfits quickly. A handful of epochs
# gives coherent-ish output AND a sensible held-out perplexity; training much
# longer would memorize the corpus and hurt generalization (we'll see PPL later).
model = create_gpt_small(vocab_size, max_seq_len=SEQ_LEN).to(device)
print(f"Model parameters: {model.count_parameters():,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

EPOCHS = 3
model.train()
losses = []
for epoch in range(EPOCHS):
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item(); n += 1
    avg = total / n
    losses.append(avg)
    print(f"epoch {epoch+1}/{EPOCHS}  loss {avg:.4f}")

model.eval()
print("\nDone. (Loss is falling — the model has learned *something*.)")

In [ ]:
# Quick sanity check using the library's built-in generate
sample = model.generate(
    torch.tensor([tokenizer.encode("ROMEO:\n")], device=device),
    max_new_tokens=120, temperature=0.8, top_k=20,
)
print(tokenizer.decode(sample[0].tolist()))

## 1. Autoregressive generation — the core loop

A decoder-only Transformer is a **next-token predictor**. Given a context of
tokens it outputs `logits` of shape `(batch, seq_len, vocab)`. For generation we
only care about the distribution at the **last position** — that is the model's
guess for "what comes next".

The loop is:

```
context = prompt
repeat:
    logits = model(context)[:, -1, :]   # last-position logits, shape (B, vocab)
    next_token = pick_from(logits)      # the DECODING STRATEGY lives here
    context = concat(context, next_token)
```

Two practical details:

- **Context cropping.** The model has a finite `max_seq_len` (here 64). Once the
  context grows past it, we must crop to the last `max_seq_len` tokens, or the
  position embeddings run out of range.
- **The "pick" step is everything.** Greedy, sampling, top-k, top-p, beam search —
  they all differ *only* in how they turn last-position logits into the next token.
  That is what the rest of this notebook dissects.

In [ ]:
@torch.no_grad()
def next_token_logits(model, idx):
    """Run the model and return last-position logits, shape (B, vocab).

    Crops context to max_seq_len so position embeddings stay in range."""
    idx_cropped = idx[:, -model.max_seq_len:]
    logits, _ = model(idx_cropped)
    return logits[:, -1, :]

# Encode a shared prompt we'll reuse for all strategy comparisons.
PROMPT = "ROMEO:\n"
prompt_ids = torch.tensor([tokenizer.encode(PROMPT)], device=device)
print("Prompt:", repr(PROMPT), "->", prompt_ids.tolist())

logits = next_token_logits(model, prompt_ids)
print("Last-position logits shape:", tuple(logits.shape))
probs = F.softmax(logits, dim=-1)
topv, topi = probs[0].topk(5)
print("\nTop-5 candidates for the next character:")
for p, i in zip(topv.tolist(), topi.tolist()):
    print(f"  {repr(tokenizer.decode([i])):>6}  p={p:.3f}")

## 2. Decoding strategies, from scratch

Each strategy is a small function that maps **last-position logits** to a chosen
token id. We then wrap them in one generic generation loop so the only thing that
changes between demos is the sampler.

In [ ]:
def greedy(logits):
    """Argmax — deterministic. Always picks the single most likely token."""
    return torch.argmax(logits, dim=-1, keepdim=True)


def temperature_sample(logits, temperature=1.0):
    """Divide logits by T before softmax.
    T<1 sharpens (more greedy), T>1 flattens (more random)."""
    probs = F.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1)


def top_k_sample(logits, k=10, temperature=1.0):
    """Keep only the k highest logits, renormalize over them, then sample."""
    logits = logits / temperature
    v, _ = torch.topk(logits, min(k, logits.size(-1)))
    logits = logits.masked_fill(logits < v[:, [-1]], float('-inf'))
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


def top_p_sample(logits, p=0.9, temperature=1.0):
    """Nucleus sampling: keep the smallest set of tokens whose cumulative
    probability >= p, renormalize, then sample."""
    logits = logits / temperature
    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
    cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    # Mask tokens once cumulative prob has *already* exceeded p.
    remove = cum_probs - F.softmax(sorted_logits, dim=-1) >= p
    sorted_logits = sorted_logits.masked_fill(remove, float('-inf'))
    # Scatter back to original vocab order.
    filtered = torch.full_like(logits, float('-inf'))
    filtered.scatter_(-1, sorted_idx, sorted_logits)
    probs = F.softmax(filtered, dim=-1)
    return torch.multinomial(probs, num_samples=1)

### A generic generation loop

This loop takes a `step_fn(logits) -> next_token` so we can drop in any strategy.

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, step_fn):
    """Generic autoregressive loop. `step_fn` maps last-position logits -> next id."""
    idx = idx.clone()
    for _ in range(max_new_tokens):
        logits = next_token_logits(model, idx)
        nxt = step_fn(logits)
        idx = torch.cat([idx, nxt], dim=1)
    return idx


def run(step_fn, n=160, seed=0):
    torch.manual_seed(seed)  # fix RNG so sampling demos are reproducible
    out = generate(model, prompt_ids, n, step_fn)
    return tokenizer.decode(out[0].tolist())


def show(title, txt):
    print("=" * 64)
    print(title)
    print("-" * 64)
    print(txt)


### Greedy vs temperature

**Greedy** is deterministic and locally optimal, but it has no way to escape a
loop: once the model's favorite continuation cycles back on itself, greedy repeats
it forever. Watch for repetition below.

**Temperature** rescales logits before softmax. Low T concentrates mass on the
top tokens (closer to greedy); high T flattens the distribution toward uniform
(more surprising, eventually incoherent).

In [ ]:
show("GREEDY (argmax) — note the repetition / loops", run(greedy))

for T in [0.5, 0.8, 1.2]:
    show(f"TEMPERATURE T={T}", run(lambda l: temperature_sample(l, temperature=T), seed=1))

### Top-k vs top-p (nucleus)

**Top-k** keeps a *fixed* number of candidates. Its weakness: `k` is constant
regardless of how peaked or flat the distribution is. When the model is very
confident, a large `k` lets in junk; when it is unsure, a small `k` cuts off good
options.

**Top-p / nucleus** keeps the *smallest set of tokens whose cumulative probability
reaches `p`*. This set **adapts**: tiny when the model is confident, large when it
is uncertain. That adaptivity is why nucleus sampling tends to beat top-k for
open-ended generation.

In [ ]:
for k in [5, 20]:
    show(f"TOP-K k={k}", run(lambda l: top_k_sample(l, k=k, temperature=0.9), seed=2))

for p in [0.8, 0.95]:
    show(f"TOP-P p={p}", run(lambda l: top_p_sample(l, p=p, temperature=0.9), seed=2))

### Repetition penalty

Even with sampling, small models love to fall into loops (`"the the the"`).
A **repetition penalty** discourages tokens that have already appeared by dividing
their logits by a factor `> 1` (for positive logits) — equivalently pushing them
toward `-inf`. This is the CTRL-paper trick.

Below we compare greedy *with* and *without* the penalty; the penalty visibly
breaks up loops.

In [ ]:
def repetition_penalty_step(model, idx, penalty=1.3, base_step=greedy):
    """Apply a repetition penalty to already-generated tokens, then decode."""
    logits = next_token_logits(model, idx)
    # Gather logits of tokens already in the context.
    for b in range(idx.size(0)):
        seen = torch.unique(idx[b])
        vals = logits[b, seen]
        # Standard CTRL rule: divide positive logits, multiply negative ones.
        vals = torch.where(vals > 0, vals / penalty, vals * penalty)
        logits[b, seen] = vals
    return base_step(logits)


@torch.no_grad()
def generate_with_penalty(model, idx, max_new_tokens, penalty=1.3):
    idx = idx.clone()
    for _ in range(max_new_tokens):
        nxt = repetition_penalty_step(model, idx, penalty=penalty)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

show("GREEDY, no penalty", run(greedy))
out = generate_with_penalty(model, prompt_ids, 160, penalty=1.4)
show("GREEDY + repetition penalty=1.4", tokenizer.decode(out[0].tolist()))

### Beam search

Greedy/sampling commit to one token at a time. **Beam search** keeps the top-`B`
*partial sequences* (beams) ranked by summed log-probability. At each step it
expands every beam by every vocab token, then prunes back to the best `B`.

- **Pro:** finds higher-probability *sequences* than greedy — great for
  **low-entropy** tasks like machine translation or summarization where there is
  essentially one correct answer.
- **Con:** for open-ended text it produces bland, repetitive, "safe" output and
  collapses diversity, so it is rarely used for creative generation.

We keep the beam width and length small to stay fast.

In [ ]:
@torch.no_grad()
def beam_search(model, idx, max_new_tokens, beam_width=3):
    """Minimal beam search. Tracks (sequence, cumulative_logprob) for B beams."""
    # Each beam: (tensor of ids shape (1, T), score). Start with one beam.
    beams = [(idx.clone(), 0.0)]
    for _ in range(max_new_tokens):
        candidates = []
        for seq, score in beams:
            logits = next_token_logits(model, seq)
            logprobs = F.log_softmax(logits, dim=-1)[0]  # (vocab,)
            top_lp, top_id = logprobs.topk(beam_width)
            for lp, tid in zip(top_lp.tolist(), top_id.tolist()):
                new_seq = torch.cat([seq, torch.tensor([[tid]], device=seq.device)], dim=1)
                candidates.append((new_seq, score + lp))
        # Prune to the best B beams by total log-prob.
        candidates.sort(key=lambda c: c[1], reverse=True)
        beams = candidates[:beam_width]
    best_seq, best_score = beams[0]
    return best_seq, best_score

for B in [1, 3]:
    seq, score = beam_search(model, prompt_ids, 80, beam_width=B)
    show(f"BEAM SEARCH width={B}  (total logprob {score:.1f})",
         tokenizer.decode(seq[0].tolist()))

### Side-by-side summary

Same model, same prompt, different decoders. The table summarizes the trade-offs;
the cell above gives you the actual text to compare.

In [ ]:
summary = [
    ("Greedy",            "Deterministic; fast; prone to loops/repetition"),
    ("Temperature",       "Tunes randomness; T<1 safe, T>1 wild"),
    ("Top-k",             "Fixed candidate count; simple but non-adaptive"),
    ("Top-p (nucleus)",   "Adaptive candidate set; best default for open text"),
    ("Repetition penalty","Add-on that breaks loops; combine with the above"),
    ("Beam search",       "Best for low-entropy tasks (MT); bland for open text"),
]
print(f"{'Strategy':<20} | Notes")
print("-" * 70)
for name, note in summary:
    print(f"{name:<20} | {note}")

## 3. The efficiency problem

The naive loop calls `model(context)` on the **entire growing context** at every
step. To generate token `t` we process `t` tokens. Over `n` generated tokens the
total work is `1 + 2 + ... + n ≈ n²/2` — **quadratic**.

Worse, almost all of that work is *redundant*: when we append one token, the
model recomputes the keys, values, and hidden states for *every previous token*,
even though they are identical to last step.

Let's measure how per-step cost grows with context length.

In [ ]:
@torch.no_grad()
def time_forward(model, length, repeats=5):
    """Average wall-clock for a single forward pass over `length` tokens."""
    idx = torch.randint(0, vocab_size, (1, length), device=device)
    # warmup
    model(idx)
    t0 = time.perf_counter()
    for _ in range(repeats):
        model(idx)
    return (time.perf_counter() - t0) / repeats

lengths = [4, 8, 16, 32, 64]
times = [time_forward(model, L) * 1000 for L in lengths]  # ms

print(f"{'context len':>12} | {'forward (ms)':>12}")
print("-" * 28)
for L, t in zip(lengths, times):
    print(f"{L:>12} | {t:>12.3f}")

plt.figure(figsize=(7, 4))
plt.plot(lengths, times, 'o-', lw=2)
plt.xlabel('context length (tokens)')
plt.ylabel('single forward pass (ms)')
plt.title('Per-step cost grows with context — naive generation is O(n²) total')
plt.grid(alpha=0.3)
plt.show()

## 4. The KV cache — the centerpiece

### Why it works

In **causal** self-attention, token `i` can only attend to tokens `j <= i`. The
key and value vectors for a past token depend *only on that token and the ones
before it* — they are **independent of any token appended later**. So when we add
a new token, the K/V of all previous tokens are *unchanged*.

A **KV cache** exploits this: we store the keys and values computed for every past
token, and at each generation step we

1. compute Q, K, V **only for the single new token**,
2. **append** its K, V to the cache,
3. attend the new token's Q against the *entire* cached K, V.

This turns each step from O(context) into O(1) work for the new token (attention
is still O(context) but the expensive projections/FFN run on one token instead of
the whole sequence). Total generation drops from O(n²) toward O(n).

We will **not** modify `src/`. Instead we build a minimal, self-contained cached
multi-head causal-attention model right here and verify it matches the recompute
path, then benchmark it.

In [ ]:
class CachedMHA(nn.Module):
    """Multi-head causal self-attention with an optional incremental KV cache."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cache=None):
        """x: (B, T, d_model).
        If cache is None: full causal attention (training / first pass).
        If cache is a dict: incremental mode — x is the NEW tokens only,
        past keys/values come from cache['k'], cache['v']."""
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, n_heads, T, d_head)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        if cache is not None:
            if cache.get('k') is not None:
                k = torch.cat([cache['k'], k], dim=2)  # append along time
                v = torch.cat([cache['v'], v], dim=2)
            cache['k'], cache['v'] = k, v

        # scores: (B, n_heads, Tq, Tk)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        Tq, Tk = scores.size(-2), scores.size(-1)
        # Causal mask: query position (offset by past) may see keys up to itself.
        offset = Tk - Tq
        qpos = torch.arange(Tq, device=x.device).view(-1, 1) + offset
        kpos = torch.arange(Tk, device=x.device).view(1, -1)
        scores = scores.masked_fill(kpos > qpos, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = attn @ v  # (B, n_heads, Tq, d_head)
        out = out.transpose(1, 2).contiguous().view(B, Tq, C)
        return self.proj(out)


class TinyLM(nn.Module):
    """Tiny decoder-only LM: embed -> [attn + MLP]xL -> head. Supports KV cache."""
    def __init__(self, vocab, d_model=64, n_heads=4, n_layers=2, max_len=256):
        super().__init__()
        self.max_len = max_len
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            nn.ModuleDict({
                'ln1': nn.LayerNorm(d_model),
                'attn': CachedMHA(d_model, n_heads),
                'ln2': nn.LayerNorm(d_model),
                'mlp': nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                     nn.Linear(4 * d_model, d_model)),
            }) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)

    def forward(self, idx, caches=None, pos_offset=0):
        B, T = idx.shape
        positions = torch.arange(pos_offset, pos_offset + T, device=idx.device)
        x = self.tok(idx) + self.pos(positions)[None]
        for i, blk in enumerate(self.blocks):
            cache = None if caches is None else caches[i]
            x = x + blk['attn'](blk['ln1'](x), cache=cache)
            x = x + blk['mlp'](blk['ln2'](x))
        return self.head(self.ln_f(x))

    def empty_caches(self):
        return [{'k': None, 'v': None} for _ in self.blocks]

tiny = TinyLM(vocab_size, max_len=512).to(device).eval()
print(f"TinyLM params: {sum(p.numel() for p in tiny.parameters()):,}")

### Correctness check

A cache is only useful if it produces **identical** results to the naive recompute
path. We generate greedily both ways from the same prompt and assert the token
sequences match exactly. (We use random weights here — correctness does not depend
on training.)

In [ ]:
@torch.no_grad()
def generate_naive(model, idx, n):
    """Recompute the full forward over the whole context every step."""
    idx = idx.clone()
    for _ in range(n):
        logits = model(idx[:, -model.max_len:])
        nxt = logits[:, -1, :].argmax(-1, keepdim=True)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

@torch.no_grad()
def generate_cached(model, idx, n):
    """Process the prompt once to fill the cache, then feed one token at a time."""
    idx = idx.clone()
    caches = model.empty_caches()
    # Prime the cache with the full prompt.
    logits = model(idx, caches=caches, pos_offset=0)
    nxt = logits[:, -1, :].argmax(-1, keepdim=True)
    idx = torch.cat([idx, nxt], dim=1)
    for step in range(1, n):
        offset = idx.size(1) - 1  # position of the single new token
        logits = model(nxt, caches=caches, pos_offset=offset)
        nxt = logits[:, -1, :].argmax(-1, keepdim=True)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

start = torch.tensor([tokenizer.encode("To be ")], device=device)
a = generate_naive(tiny, start, 40)
b = generate_cached(tiny, start, 40)
print("naive :", a[0].tolist()[:20], "...")
print("cached:", b[0].tolist()[:20], "...")
assert torch.equal(a, b), "KV cache output diverged from naive recompute!"
print("\nMATCH: KV-cached generation is bit-for-bit identical to naive recompute.")

### Benchmark: naive recompute vs KV cache

Now the payoff. We generate `N` tokens both ways and compare wall-clock. To make
the quadratic effect visible on a CPU we use a longer-context TinyLM and a
moderate token count.

In [ ]:
def benchmark(model, prompt_ids, n, reps=3):
    best_naive, best_cached = float('inf'), float('inf')
    for _ in range(reps):
        t0 = time.perf_counter(); generate_naive(model, prompt_ids, n)
        best_naive = min(best_naive, time.perf_counter() - t0)
        t0 = time.perf_counter(); generate_cached(model, prompt_ids, n)
        best_cached = min(best_cached, time.perf_counter() - t0)
    return best_naive, best_cached

bench_prompt = torch.tensor([tokenizer.encode("ROMEO:")], device=device)
N_TOKENS = 160
naive_t, cached_t = benchmark(tiny, bench_prompt, N_TOKENS, reps=2)

print(f"Generating {N_TOKENS} tokens:")
print(f"  naive recompute : {naive_t*1000:8.1f} ms")
print(f"  KV cache        : {cached_t*1000:8.1f} ms")
print(f"  speedup         : {naive_t / cached_t:8.2f}x")

plt.figure(figsize=(6, 4))
plt.bar(['naive\nrecompute', 'KV cache'], [naive_t*1000, cached_t*1000],
        color=['#c44', '#4a4'])
plt.ylabel('time (ms)')
plt.title(f'Generating {N_TOKENS} tokens — {naive_t/cached_t:.1f}x speedup')
for i, v in enumerate([naive_t*1000, cached_t*1000]):
    plt.text(i, v, f'{v:.0f} ms', ha='center', va='bottom')
plt.show()

### How time scales with sequence length

The gap *widens* as you generate more tokens, because naive cost grows
quadratically while cached cost grows roughly linearly.

In [ ]:
ns = [40, 80, 120, 160]
naive_times, cached_times = [], []
for n in ns:
    nt, ct = benchmark(tiny, bench_prompt, n, reps=1)
    naive_times.append(nt * 1000); cached_times.append(ct * 1000)

plt.figure(figsize=(7, 4))
plt.plot(ns, naive_times, 'o-', label='naive (O(n²))', color='#c44')
plt.plot(ns, cached_times, 'o-', label='KV cache (~O(n))', color='#4a4')
plt.xlabel('tokens generated'); plt.ylabel('total time (ms)')
plt.title('Naive cost curves up; KV-cache stays near-linear')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print("speedup at each length:")
for n, nt, ct in zip(ns, naive_times, cached_times):
    print(f"  {n:>4} tokens: {nt/ct:.2f}x")

## 5. Production inference, briefly

The KV cache is step one. Real serving stacks layer on more:

- **Batching & padding for throughput.** Serve many requests in one forward pass.
  Sequences have different lengths, so they are **padded** to a common length and an
  **attention mask** hides the padding. Bigger batches = higher GPU utilization and
  throughput, at the cost of waiting to fill a batch. *Continuous batching* (vLLM)
  adds/removes sequences mid-flight to avoid stragglers.

- **Latency vs throughput.** These trade off. **Latency** = time to first/next token
  for one user (interactive chat cares about this). **Throughput** = tokens/sec
  across all users (cost-efficiency cares about this). Large batches help throughput
  but can hurt per-request latency.

- **Quantization (int8 / int4).** Store weights (and sometimes the KV cache) in 8- or
  4-bit instead of 16-bit. Cuts memory and bandwidth ~2–4x with small quality loss —
  the usual way to fit a big model on a small GPU.

- **Speculative decoding.** A small, cheap "draft" model proposes several tokens
  ahead; the big "target" model then verifies them all in a *single* parallel forward
  pass, accepting the longest correct prefix. Because verification is parallel, you
  get multiple tokens for roughly the cost of one big-model step — a 2–3x speedup with
  *identical* output distribution to the target model.

- **Shrinking the KV cache (GQA / MQA).** The KV cache is often the memory bottleneck
  for long contexts. **Multi-Query Attention** shares one K/V head across all query
  heads; **Grouped-Query Attention** shares K/V across small groups — both dramatically
  shrink the cache. We cover these in **`11_modern_architectures.ipynb`**.

## 6. Evaluating generation: perplexity

How do we measure a language model? The standard intrinsic metric is **perplexity**:
the exponential of the average per-token cross-entropy on held-out text.

$$\text{PPL} = \exp\!\left(\frac{1}{N}\sum_{i=1}^{N} -\log p(x_i \mid x_{<i})\right)$$

Intuitively, perplexity is the model's "effective branching factor" — *on average,
how many equally-likely tokens is it choosing among?* Lower is better. A perfect
model that always assigns probability 1 to the truth has PPL = 1; a uniform model
over a vocab of size V has PPL = V.

We compute it on a held-out tail of the text the model did **not** train on
heavily, by averaging the model's own cross-entropy loss.

In [ ]:
@torch.no_grad()
def perplexity(model, ids, seq_len):
    """Average cross-entropy -> perplexity over non-overlapping windows."""
    model.eval()
    losses = []
    for start in range(0, len(ids) - seq_len - 1, seq_len):
        x = ids[start:start + seq_len].unsqueeze(0).to(device)
        y = ids[start + 1:start + seq_len + 1].unsqueeze(0).to(device)
        _, loss = model(x, y)
        losses.append(loss.item())
    avg = sum(losses) / len(losses)
    return math.exp(avg), avg

heldout = torch.tensor(tokenizer.encode(heldout_text), dtype=torch.long)  # unseen tail

ppl, ce = perplexity(model, heldout, SEQ_LEN)
print(f"Held-out cross-entropy: {ce:.4f} nats/token")
print(f"Held-out perplexity   : {ppl:.2f}")
print(f"(Uniform baseline over {vocab_size} chars would be PPL = {vocab_size})")

### Perplexity is not generation quality

Lower perplexity *correlates* with a better model, but it is **not** the same as
good generation:

- Perplexity scores the model against the *reference* text token-by-token. It says
  nothing about whether free-running generation is coherent, factual, or non-repetitive.
- Two models with similar perplexity can produce wildly different samples depending
  on the **decoding strategy** (sections 2!). Decoding choices don't change perplexity
  at all.
- Perplexity is not comparable across different tokenizers/vocabularies.

That's why real evaluation also uses:

- **Automatic task metrics** — BLEU/ROUGE (translation, summarization), exact-match/F1
  (QA), pass@k (code).
- **Model-based & human eval** — LLM-as-a-judge, pairwise preference ratings, and
  human annotation for helpfulness/coherence. For open-ended chat, human preference is
  still the gold standard.

## Summary

We dissected the **inference** half of a language model — turning next-token
distributions into text, efficiently.

- **Autoregressive loop:** feed context, take last-position logits, pick a token,
  append, repeat — cropping to `max_seq_len`.
- **Decoding strategies** (implemented from scratch): greedy, temperature, top-k,
  top-p/nucleus, repetition penalty, beam search. The "pick" step is where they all
  differ. Nucleus adapts its candidate set; beam search shines on low-entropy tasks
  but is bland for open text.
- **Efficiency:** naive generation is O(n²) because it recomputes the whole context
  every step.
- **KV cache:** causal attention means past keys/values never change, so we cache
  them and process only the new token. We built a self-contained cached attention
  model, verified it is **bit-identical** to recompute, and **benchmarked the speedup**.
- **Production:** batching/padding, latency vs throughput, quantization, speculative
  decoding, and GQA/MQA to shrink the cache.
- **Evaluation:** perplexity measures model fit but is *not* generation quality;
  real eval needs task metrics and human/LLM judgment.

### Key Takeaways

1. The decoder is a next-token predictor; **decoding strategy** decides the output, and
   it is independent of the trained weights.
2. **Top-p/nucleus** is the sensible default for open-ended text; **beam search** for
   low-entropy tasks like translation.
3. The **KV cache** is the single most important inference optimization — same outputs,
   roughly O(n) instead of O(n²).
4. Most serving speedups (quantization, speculative decoding, MQA/GQA) preserve output
   quality while reducing memory/compute.
5. **Perplexity ≠ quality.** Low perplexity is necessary but not sufficient for good
   generation.

### Where next

- **`10_encoder_and_seq2seq.ipynb`** — encoder-decoder models and cross-attention,
  where beam search really earns its keep (translation, summarization).
- **`11_modern_architectures.ipynb`** — RoPE, RMSNorm, SwiGLU, and **GQA/MQA** for
  shrinking the KV cache in modern LLMs.